# Fine-tuning & Prompt Optimization

#### Fine-tuning vs Prompting — The Decision

PROMPTING (what you've done so far):
  ✅ No training needed
  ✅ Change behavior instantly
  ✅ Works with any model
  ❌ Limited by base model knowledge
  ❌ Long prompts = expensive/slow
  ❌ Inconsistent on edge cases

FINE-TUNING:
  ✅ Model learns YOUR style/format
  ✅ Shorter prompts needed
  ✅ Better on domain-specific tasks
  ✅ Faster inference (smaller prompts)
  ❌ Takes time and data to train
  ❌ Need 100-1000+ examples
  ❌ Expensive to retrain often

WHEN TO FINE-TUNE:
  ✅ Consistent output format always needed
  ✅ Domain-specific language (medical, legal)
  ✅ Style matching (your company's tone)
  ✅ Task prompting costs > fine-tuning cost
  ❌ You have < 100 examples
  ❌ Task changes frequently
  ❌ General Q&A (prompting works fine)

In [1]:
# Cell 1: Imports
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import json, os, time
from typing import List, Dict, Optional
from dotenv import load_dotenv

load_dotenv()
llm  = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
fast = ChatGroq(model="llama-3.1-8b-instant",    temperature=0)

print("✅ Ready")

d:\AI\Langchain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Ready


#### Prepare Fine-tuning Dataset

In [2]:
# Cell 2: Create a training dataset
# Format: instruction → response pairs
# We'll create a dataset for a customer support bot

training_examples = [
    {
        "instruction": "Customer says: My order hasn't arrived yet. Order #12345",
        "response": "I apologize for the delay. Let me check order #12345 for you right away. Could you please confirm your email address so I can pull up the details and provide you with an accurate update on your delivery status?"
    },
    {
        "instruction": "Customer says: I want to return this product",
        "response": "I'd be happy to help you with your return. Our return policy allows returns within 30 days of purchase. Please provide your order number and reason for return, and I'll initiate the process immediately."
    },
    {
        "instruction": "Customer says: The product is broken",
        "response": "I'm sorry to hear the product arrived damaged. This is not the experience we want for our customers. Please share a photo of the damage and your order number, and we'll arrange a replacement or full refund right away."
    },
    {
        "instruction": "Customer says: How do I track my order?",
        "response": "You can track your order in three ways: (1) Check the tracking link in your confirmation email, (2) Log into your account and visit 'My Orders', or (3) Reply here with your order number and I'll get the latest update for you."
    },
    {
        "instruction": "Customer says: I was charged twice",
        "response": "I sincerely apologize for the double charge — that should never happen. I'm escalating this to our billing team immediately. Please share your order number and the last 4 digits of the card charged, and we'll process a refund within 3-5 business days."
    },
    {
        "instruction": "Customer says: Cancel my subscription",
        "response": "I understand you'd like to cancel. Before I process that, may I ask what prompted this decision? If there's something we can improve, I'd love to help. If you'd still like to proceed, I can cancel immediately with no further charges."
    },
    {
        "instruction": "Customer says: The app keeps crashing",
        "response": "I'm sorry the app is giving you trouble. Let's fix this quickly. Could you tell me: (1) Your device model and OS version, (2) When the crashes started, (3) What you were doing when it crashed? This will help me get you the right solution."
    },
    {
        "instruction": "Customer says: I never received my refund",
        "response": "I apologize for this delay. Refunds typically take 5-7 business days to appear. Let me check the status of your refund right now. Could you share your order number and the email associated with your account?"
    },
]

# Convert to Alpaca format (standard fine-tuning format)
def to_alpaca_format(examples: List[Dict]) -> List[Dict]:
    """Convert to standard Alpaca instruction format"""
    formatted = []
    for ex in examples:
        formatted.append({
            "instruction": ex["instruction"],
            "input":       "",   # no additional input
            "output":      ex["response"],
            "text": f"""Below is an instruction that describes a task.
Write a response that appropriately completes the request.

### Instruction:
{ex['instruction']}

### Response:
{ex['response']}"""
        })
    return formatted


alpaca_data = to_alpaca_format(training_examples)

# Save dataset
os.makedirs("fine_tune_data", exist_ok=True)
with open("fine_tune_data/customer_support.json", "w") as f:
    json.dump(alpaca_data, f, indent=2)

print(f"✅ Dataset created: {len(alpaca_data)} examples")
print(f"   Format: Alpaca instruction format")
print(f"   Saved:  fine_tune_data/customer_support.json")
print(f"\nSample:")
print(alpaca_data[0]["text"][:300])

✅ Dataset created: 8 examples
   Format: Alpaca instruction format
   Saved:  fine_tune_data/customer_support.json

Sample:
Below is an instruction that describes a task.
Write a response that appropriately completes the request.

### Instruction:
Customer says: My order hasn't arrived yet. Order #12345

### Response:
I apologize for the delay. Let me check order #12345 for you right away. Could you please confirm your e


#### QLoRA Fine-tuning on RTX 5060

In [3]:
# Cell 3: QLoRA setup - quantized LoRA for consumer GPUs

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)
from trl import SFTTrainer
from datasets import Dataset

# Check GPU
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU — will use CPU (slower)")

GPU available: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM: 8.5 GB


In [4]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
print(f"CUDA capability: sm_{torch.cuda.get_device_capability(0)[0]}{torch.cuda.get_device_capability(0)[1]}")
print(f"GPU:             {torch.cuda.get_device_name(0)}")
print(f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Quick compute test
x = torch.tensor([1.0, 2.0, 3.0]).cuda()
print(f"\nGPU tensor test: {x * 2}")
print("✅ RTX 5060 working with PyTorch!")

PyTorch version: 2.12.0.dev20260408+cu128
CUDA available:  True
CUDA capability: sm_120
GPU:             NVIDIA GeForce RTX 5060 Laptop GPU
VRAM:            8.5 GB

GPU tensor test: tensor([2., 4., 6.], device='cuda:0')
✅ RTX 5060 working with PyTorch!


In [5]:
# Cell 4: QLoRA configuration
# QLoRA = Quantized LoRA = fine-tune big models on small GPUs

# ── Step 1: Quantization config ───────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                       # load model in 4-bit
    bnb_4bit_quant_type="nf4",               # NF4 quantization
    bnb_4bit_compute_dtype=torch.float16,    # compute in fp16
    bnb_4bit_use_double_quant=True           # double quantization
)

# ── Step 2: LoRA config ───────────────────────────────────────
lora_config = LoraConfig(
    r=16,                       # rank - higher = more params = better quality
    lora_alpha=32,              # scaling factor (usually 2x rank)
    target_modules=[            # which layers to train
        "q_proj", "k_proj",
        "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,          # regularization
    bias="none",                # don't train bias
    task_type=TaskType.CAUSAL_LM
)

print("QLoRA Configuration:")
print(f"""
Quantization:
  bits:          4-bit NF4
  compute dtype: float16
  double quant:  yes

LoRA:
  rank (r):      16
  alpha:         32
  dropout:       0.05
  target:        attention + MLP layers

Memory estimate for Llama-3.2-1B:
  Without QLoRA: ~4GB VRAM
  With QLoRA:    ~1.5GB VRAM  ← your RTX 5060 handles this easily
""")

QLoRA Configuration:

Quantization:
  bits:          4-bit NF4
  compute dtype: float16
  double quant:  yes

LoRA:
  rank (r):      16
  alpha:         32
  dropout:       0.05
  target:        attention + MLP layers

Memory estimate for Llama-3.2-1B:
  Without QLoRA: ~4GB VRAM
  With QLoRA:    ~1.5GB VRAM  ← your RTX 5060 handles this easily

